<a href="https://colab.research.google.com/github/your-org/alexpose/blob/main/experiments/multiple-sclerosis/02_anatomical_mask_and_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Masking and tokenization

S-JEPA learns by hiding part of a skeleton sequence and predicting it in feature space. The token and mask operations apply to either dataset version; the dataset changes the number of windows and source groups, not the token definition. Run notebook 01 first to build the versioned `keypoints-full/` cache.

> **What changed, and why.** An earlier version of this project hid the *same* twelve clinical joints on every single step. That turned out to be a real bug: the encoder never saw those joints as context, so their internal position settings received no learning signal, yet the classifier then pooled exactly those joints. We now use **stochastic graph-time masks**: a different connected group of joints is hidden each step, so every joint is sometimes context and sometimes a target. Clinical knowledge still guides us, but gently, by choosing the leg and shoulder joints as targets a bit more often. We also do **not** bias toward the busiest joints (the paper's motion-aware masking), because in MS and PD the telling sign is often *reduced* motion, which a high-motion mask would hide.


In [ ]:
# --- Setup: install dependencies (Colab installs; local usually already has them) ---
import importlib, importlib.util, subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

def _need(mod):
    return importlib.util.find_spec(mod) is None

# Light deps used by every notebook.
_pkgs = []
for mod, pip_name in [('cv2','opencv-python'), ('mediapipe','mediapipe'),
                      ('sklearn','scikit-learn'), ('pandas','pandas'),
                      ('matplotlib','matplotlib'), ('tqdm','tqdm')]:
    if _need(mod):
        _pkgs.append(pip_name)
if _pkgs:
    print('installing:', _pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs])
else:
    print('all light dependencies already present')

In [ ]:
# --- Make `sjepa` and `ambient` importable, locally and in Colab ---
from pathlib import Path
import sys, subprocess

def _find_exp_dir():
    # Local run: this notebook sits in experiments/multiple-sclerosis.
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'sjepa' / '__init__.py').exists():
            return p
    return None

EXP_DIR = _find_exp_dir()
if EXP_DIR is None:
    # Colab: clone the repo, then point at the experiment folder.
    REPO = 'https://github.com/your-org/alexpose.git'  # <-- edit to your fork
    if not Path('alexpose').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO])
    EXP_DIR = Path('alexpose') / 'experiments' / 'multiple-sclerosis'

REPO_ROOT = EXP_DIR.parents[1]
for p in (str(EXP_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('experiment dir:', EXP_DIR)
print('repo root     :', REPO_ROOT)

In [ ]:
# --- Paths and profile (reads the root .env if python-dotenv is present) ---
import os
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass

VIDEO_DIR = EXP_DIR / 'video-data-full'
ARTIFACT_DIR = EXP_DIR / 'artifacts'
KEYPOINTS_DIR = ARTIFACT_DIR / 'keypoints-full'
IMAGES_DIR = EXP_DIR / 'images'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Pick the model size profile. 'laptop' is the fast default; set SJEPA_PROFILE=gpu
# in your .env for a larger model, or SJEPA_SMOKE=1 for a near-instant test run.
os.environ.setdefault('SJEPA_PROFILE', 'laptop')
print('SJEPA_PROFILE =', os.environ['SJEPA_PROFILE'],
      '| SJEPA_SMOKE =', os.environ.get('SJEPA_SMOKE', '0'))

## Keep each source video together

We use the same frozen five-fold registry in notebooks 02–06. A source video may have several clips; each clip may yield overlapping windows. All of those relatives stay together. Each round uses about 60% of sources for training, 20% for validation, and 20% for testing. Only notebook 06 evaluates test clips.

The splitter runs on one row per source, with condition labels used to balance source counts. It never splits windows. The loader checks the full cache, reviewed exclusions, and registry checksum. A changed cache requires a new registry and new checkpoints. See [the full method](docs/11-full-data-splits.md).

In [ ]:
from IPython.display import display
import pandas as pd
from sjepa.splits import load_full_registry, partition_records, split_summary
records, registry = load_full_registry(EXP_DIR)
FOLD = 0  # teaching example; notebook 06 independently trains all five folds
train_recs, val_recs, test_recs = partition_records(records, registry, FOLD)
display(pd.DataFrame(split_summary(records, registry)))
print('usable clips:', len(records), '| excluded raw clips:', len(registry['inventory']['exclusions']))
print('registry:', registry['registry_sha256'])

## Tokenizing a window

A training window is a short movie of stick figures. We group `l = 4` adjacent frames of one joint into a single token, so each token summarizes how that joint moved over a moment. With 32 frames and 33 joints that gives `(32 / 4) x 33 = 264` tokens. Token index is `t * V + v` (time block `t`, joint `v`).


In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(IMAGES_DIR / 'tokenization.svg')))

In [ ]:
from sjepa.config import get_config, describe
cfg = get_config()  # honours SJEPA_PROFILE
print(describe(cfg))
print('tokens per window N =', cfg.num_tokens,
      f'= {cfg.num_time_tokens} time blocks x {cfg.num_joints} joints')

## The clinical joints (domain context, not a permanent mask)

The file `mapping-data/ms-pd-mapping.md` lists the joints clinicians care about for ms and pd. After removing duplicates and sorting, we get exactly twelve BlazePose landmarks: both shoulders and both complete legs. We keep this table as **domain knowledge** that biases how often a joint is chosen as a target, but every joint can still be both context and target.


In [ ]:
from sjepa.masking_v2 import CLINICAL_JOINTS
from ambient.pose.keypoint_data import MEDIAPIPE_33_NAMES
import pandas as pd

features_for = {
    11: 'shoulder_symmetry_index, trunk_lean_angle',
    12: 'shoulder_symmetry_index, trunk_lean_angle',
    23: 'walking_speed_ms, hip_asymmetry, knee_range, trunk_lean_angle',
    24: 'walking_speed_ms, hip_asymmetry, knee_range, trunk_lean_angle',
    25: 'knee_range, ankle_range', 26: 'knee_range, ankle_range',
    27: 'knee_range, ankle_range, step_width_m', 28: 'knee_range, ankle_range, step_width_m',
    29: 'stride_length_m, double_support_pct, stride_time_cv, ankle_range',
    30: 'stride_length_m, double_support_pct, stride_time_cv, ankle_range',
    31: 'stride_length_m, double_support_pct, stride_time_cv, ankle_range',
    32: 'stride_length_m, double_support_pct, stride_time_cv, ankle_range',
}
table = pd.DataFrame([
    {'BLAZEPOSE_33 index': j, 'Keypoint name': MEDIAPIPE_33_NAMES[j],
     'Features involved': features_for[j]}
    for j in sorted(CLINICAL_JOINTS)
])
table

## Stochastic graph-time masks

Each step we sample a per-example mask: connected groups of joints (a limb or the trunk) over a contiguous span of time. The cell below samples a few masks and shows they differ, that every window keeps visible context somewhere, and that over a bank of masks every joint is both visible and targeted often enough (the coverage gates). A joint can be masked in every block of one window: overlapping regions can cover its full duration. The maximum span applies to each sampled region, not their union. Hips belong to both trunk and leg regions, so their target frequency can be higher. Coverage is measured across fresh mask draws; individual frames and windows do not have a per-joint visibility guarantee. There is also a temporal bias: intervals placed entirely inside a window cover middle blocks more often than the ends. The exact-time table below exposes this; passing the window-coverage gates does not establish balanced masking or optimal training.


In [ ]:
import numpy as np
from importlib import reload
import sjepa.masking_v2 as masking
masking = reload(masking)  # pick up local edits in an existing notebook kernel
from sjepa.masking_v2 import sample_mask_batch, mask_bank_stats

rng = np.random.default_rng(0)
batch = sample_mask_batch(6, cfg.num_joints, cfg.num_time_tokens, rng)
print('mask batch shape (B, N):', batch.shape)
print('unique masks in the batch:', len({row.tobytes() for row in batch}), 'of 6')
print('every row has context and target:',
      bool((~batch).any(1).all() and batch.any(1).all()))

stats = mask_bank_stats(cfg.num_joints, cfg.num_time_tokens, n_masks=512, seed=0)
print(f'over 512 masks: min visible-at-least-once/window {stats.joint_visible_frac.min():.2f} '
      f'(gate >=0.20), min targeted-at-least-once/window {stats.joint_target_frac.min():.2f} (gate >=0.10)')
print(f'mean target fraction {stats.mean_target_frac:.2f}')
display(pd.DataFrame({
    'joint': MEDIAPIPE_33_NAMES,
    'visible token %': 100 * stats.joint_visible_token_frac,
    'windows with any context %': 100 * stats.joint_visible_frac,
    'windows fully masked for this joint %': 100 * stats.joint_always_target_frac,
}).round(1))
# 'Ever visible' can conceal low coverage in the middle of a window.
print('Hip context frequency at each exact time block (%):')
display(pd.DataFrame(100 * stats.token_visible_frac[:, [23, 24]],
                     columns=['left hip', 'right hip'],
                     index=np.arange(1, cfg.num_time_tokens + 1)).round(1))

Here is the difference drawn out: a fixed mask hides the same joints forever (left), while stochastic masks rotate which joints are hidden (right).


In [ ]:
display(SVG(filename=str(IMAGES_DIR / 'defect_mask_starvation.svg')))

## See successive mask draws on the same skeleton

Two clocks are shown separately: **mask sample** changes the independently drawn mask, while **time block** advances within a single model window. We replay the same motion for eight fresh masks from one seeded RNG, without filtering or recoloring any samples. Blue circles are visible context; red X markers are hidden prediction targets.

With the laptop profile, the first seed-0 sample hides both complete legs for the whole window. In later samples the hips become visible. An entirely red time block is also valid if context exists elsewhere in that window. The GIF repeats these eight saved samples; rerunning with the same seed reproduces them. Change `DEMO_SEED` to explore another batch.

The static timeline shows **every joint and time block at once**, even if your notebook viewer freezes the GIF. Its rows 23 and 24 show the exact hip mask bits. The helper also refreshes both older GIF filenames and writes a manifest so stale output can be detected. From a terminal, `python scripts/scripts_mask_demo.py` produces the same demo and `python scripts/scripts_mask_demo.py --check` detects overwritten artifacts.


In [ ]:
from importlib import reload
import sjepa.masking_v2 as masking
import sjepa.viz as viz
import sjepa.mask_demo as mask_demo
masking = reload(masking); viz = reload(viz)
mask_demo = reload(mask_demo)
from sjepa.data import sliding_windows
from IPython.display import Image, display

# Inspect only a training clip; no held-out motion guides the mask demo.
seq = sliding_windows(train_recs[0].load_norm(),
                      cfg.window_frames, cfg.window_stride)[0]
DEMO_SEED = 0
N_MASK_SAMPLES = 8
demo = mask_demo.write_mask_demo(seq, ARTIFACT_DIR, frame_group=cfg.frame_group,
                                fps=cfg.target_fps, seed=DEMO_SEED, n_samples=N_MASK_SAMPLES)
demo_masks = demo.masks
print('Every joint appears in both roles across these samples:',
      bool(demo_masks.any((0, 1)).all() and (~demo_masks).any((0, 1)).all()))
print(f'Each sample: {cfg.window_frames} frames; each block: {cfg.frame_group} frames.')
display(pd.DataFrame({
    'mask sample': np.arange(1, N_MASK_SAMPLES + 1),
    'left hip visible blocks': (~demo_masks[:, :, 23]).sum(1),
    'right hip visible blocks': (~demo_masks[:, :, 24]).sum(1),
    'out of blocks': cfg.num_time_tokens,
}))
mask_demo.verify_mask_demo(ARTIFACT_DIR)
display(Image(filename=str(demo.timeline)))
display(Image(filename=str(demo.animation)))